# Walkthrough: Cu interconnect size effect

A short guided tour of this repository: load the digitized data, evaluate the
Fuchs–Sondheimer (surface) and Mayadas–Shatzkes (grain-boundary) models, and
reproduce the main fit figure. Full details are in `docs/report.md`.

Run this notebook from the `notebooks/` directory (or the repo root).

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.path.abspath(".")
sys.path.insert(0, os.path.join(ROOT, "src"))
import models as M

## 1. The digitized Steinhögl (2002) wire data

Nine (linewidth, resistivity) points digitized from their Fig. 3
(Cu damascene wires, height 230 nm, 295 K). The CSV header documents the axis
calibration and its verification.

In [ ]:
rows = []
with open(os.path.join(ROOT, "data", "steinhogl2002_fig3_points.csv")) as fh:
    for line in fh:
        line = line.strip()
        if line and not line.startswith("#") and not line.startswith("w_nm"):
            rows.append([float(x) for x in line.split(",")])
arr = np.array(rows)
w, rho = arr[:, 0], arr[:, 1]
sig = 0.5 * (arr[:, 2] + arr[:, 3])
list(zip(w, rho))

## 2. Evaluate the models

`RectWireInterp` gives fast, disk-cached evaluation of the exact Chambers
surface-scattering solution for rectangular wires; the Mayadas–Shatzkes
grain-boundary factor is analytic. The combined model follows Steinhögl
Eq. (5): additive resistivity increments, with grain size d = min(w, h).
(The first run builds a cache table, ~2 min; afterwards it is instant.)

In [ ]:
RHO0, LAM, H = 1.90, 40.0, 230.0     # from the paper (see docs/PARAMETERS.md)
P_FIT, R_FIT = 0.0, 0.425            # our best fit (src/fit_steinhogl.py)

WIRE = M.RectWireInterp(cache_path=os.path.join(ROOT, "data", "cache", "chambers_p0_table.npz"))

wg = np.logspace(np.log10(35), 3, 80)
combined = [RHO0 * WIRE.combined(x, H, LAM, P_FIT, R_FIT, min(x, H)) for x in wg]
ms_only  = [RHO0 * float(M.ms_rho_ratio(LAM, min(x, H), R_FIT)) for x in wg]
fs_only  = [RHO0 * WIRE.rho_ratio(x, H, LAM, P_FIT) for x in wg]

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 4.6))
ax.errorbar(w, rho, yerr=sig, fmt="o", ms=5, mfc="white", mec="k", ecolor="k",
            capsize=2.5, label="Steinhögl 2002 (digitized)")
ax.plot(wg, combined, "-", c="tab:red", lw=1.8, label=f"combined fit (p={P_FIT}, R={R_FIT})")
ax.plot(wg, ms_only, "-.", c="tab:blue", lw=1.2, label="grain boundaries only")
ax.plot(wg, fs_only, ":", c="tab:green", lw=1.4, label="surfaces only")
ax.axhline(1.68, color="0.6", ls=":", lw=1)
ax.set_xscale("log"); ax.set_xlabel("linewidth w [nm]")
ax.set_ylabel(r"$\rho$ [$\mu\Omega\,$cm]"); ax.legend(frameon=False)
ax.set_title("Cu wire resistivity vs. linewidth")
plt.show()

The main physics is visible at a glance: surface scattering alone (green)
cannot reach the data, while grain-boundary scattering (blue) carries most of
the size effect because the grain size shrinks with the linewidth.

## 3. Where to go next

* `src/fit_steinhogl.py` — the full fit with uncertainty ranges and the p–R degeneracy.
* `src/validate_yarimbiyik.py` — the model tested on an independent thin-film dataset with **no refitting** (7.4 % mean deviation).
* `src/sensitivity.py` — which parameter matters most (answer: R, then grain size; specularity is ~6× weaker).
* `docs/report.md` — everything, with caveats and references.